# Lab Summary: Profiling Python GPU Kernels with Nsight Compute

## Key Learnings

### Profiling with Nsight Compute

- Numba and Triton kernels are fully compatible for profiling with Nsight Compute
- Use `ncu --set full` for comprehensive metrics; `-k regex:name` to filter kernels
- Line info is preserved by default, enabling source-level correlation
- The `--import-source yes` flag embeds source in reports for portable analysis
- The Nsight Jupyterlab extension enables profiling notebook cells directly

### Understanding the Profiler Report

<table style="margin-left: 0;">
    <tr><th>Section</th><th>What It Shows</th><th>Key Metrics</th></tr>
    <tr><td>GPU Speed of Light</td><td>High-level throughput</td><td>SM %, Memory %</td></tr>
    <tr><td>Compute Workload</td><td>Instruction mix</td><td>Pipeline utilization</td></tr>
    <tr><td>Memory Workload</td><td>Memory hierarchy</td><td>L1/L2 hit rates, bandwidth</td></tr>
    <tr><td>Warp State Statistics</td><td>Thread execution</td><td>Stall reasons, cycles</td></tr>
    <tr><td>Occupancy</td><td>Resource usage</td><td>Achieved vs theoretical</td></tr>
</table>

### Optimization Techniques

**Profiler guidance**: High Memory Throughput + Low Compute Throughput → memory-bound kernel. Look for opportunities to reduce memory pressure, improve access patterns and increase compute per thread.

<table style="margin-left: 0;">
    <tr><th>Stall Type</th><th>Cause</th><th>Optimization</th></tr>
    <tr><td><b>Long Scoreboard</b></td><td>Global memory latency</td><td>Tiling for reuse, FP8, pipelining</td></tr>
    <tr><td><b>Short Scoreboard</b></td><td>Shared memory / MIO</td><td>Reduce bank conflicts, async copies</td></tr>
</table>

<table style="margin-left: 0;">
    <tr><th>Technique</th><th>What Changed</th><th>Impact</th></tr>
    <tr><td><b>FP8 Precision</b></td><td>8-bit floats for Q, K, V</td><td>2× bandwidth reduction → ~1.5× speedup</td></tr>
    <tr><td><b>Tiling</b></td><td>Process BLOCK_M queries per block</td><td>K/V reuse → ~20× speedup</td></tr>
</table>

### Nsight Python

[Nsight Python](https://docs.nvidia.com/nsight-python) automates GPU kernel profiling with simple decorators:

```python
import nsight

@nsight.analyze.kernel
def run_attention():
    with nsight.annotate("attention"):
        attention_prefill_dot[(B*H, SQ)](...)
        torch.cuda.synchronize()

# Run as standalone script (not inside Jupyter)
if __name__ == "__main__":
    session = run_attention()
    session.to_dataframe()
```

Key features:
- **`@nsight.analyze.kernel`** - Decorate host functions to enable profiling
- **`nsight.annotate`** - Context manager for marking kernel launches
- **Standalone scripts** - Must run as Python scripts (not inside Jupyter notebooks)
- **Multi-variant comparison** - Profile multiple kernel variants in one script
- **Automatic visualizations** - Generate charts and roofline plots

## Further Learning

### Documentation

- [Nsight Compute Documentation](https://docs.nvidia.com/nsight-compute">docs.nvidia.com/nsight-compute)
- [Nsight Compute CLI Reference](https://docs.nvidia.com/nsight-compute/NsightComputeCli/index.html">NsightComputeCli)
- [Profiling Guide](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html">ProfilingGuide)
- [Metrics Reference](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#metrics-reference)
- [Python Report Interface](https://docs.nvidia.com/nsight-compute/PythonReportInterface/index.html">ProfilingGuide)
- [Nsight Python](https://docs.nvidia.com/nsight-python">docs.nvidia.com/nsight-python)

### Tutorials & Training

<table style="margin-left: 0;">
    <tr><td><a href="https://github.com/NVIDIA/nsight-python">Nsight Python GitHub</a></td><td>Source code, examples, and issue tracker</td></tr>
    <tr><td><a href="https://github.com/NVIDIA/nsight-training">Nsight Training Repository</a></td><td>Sample code, reports, and Jupyter notebooks</td></tr>
    <tr><td><a href="https://developer.nvidia.com/nsight-compute-blogs">Nsight Compute Blog</a></td><td>Feature deep-dives and use cases</td></tr>
    <tr><td><a href="https://developer.nvidia.com/nsight-compute-videos">Nsight Compute Videos</a></td><td>Video tutorials and GTC presentations</td></tr>
    <tr><td><a href="https://www.nvidia.com/en-us/training/">DLI Courses</a></td><td>NVIDIA Deep Learning Institute courses</td></tr>
</table>

---

## Thank You!

We hope this lab has given you practical skills for profiling and optimizing GPU kernels with Nsight Compute. The techniques demonstrated here apply broadly to any CUDA workload—whether written in C++, Python, Triton, or other frameworks.

**Questions or feedback?** Visit the [NVIDIA Developer Forums](https://forums.developer.nvidia.com/) or reach out to your instructor.

[← Back to Introduction](../lab.ipynb)